### Imports

In [24]:
import json
import os
import pandas as pd

from utils_MS import *

# %load_ext autotime

In [25]:
# def run(params):

### Parameters

In [26]:
""" try:
    dir = os.path.dirname(os.path.abspath(__file__))
except:
    dir = os.getcwd()
print(dir) """

' try:\n    dir = os.path.dirname(os.path.abspath(__file__))\nexcept:\n    dir = os.getcwd()\nprint(dir) '

In [27]:
dict_dataset = {
    1: ["Mentos_2_process_NormalizationFiltered_format", ["Orange"]], # new mentos,  for Metabolomics
    2: ["deybis_filter_september_2br_3ar_format", ["SecoAmazonas"]],
    3: ["deybis_filter_december_2br_3ar_format", ["SecoAmazonas"]],
    4: ["deybis_filter_september_2br_10ar_format", ["SecoAmazonas"]],
    5: ["deybis_filter_december_2br_10ar_format", ["SecoAmazonas"]], # for Metabolomics
    6: ["deybis_filter_september_min_2br_3ar_format", ["SecoAmazonas"]],
    7: ["deybis_filter_december_min_2br_3ar_format", ["SecoAmazonas"]],
    8: ["vanessa_december_2br_3ar_format", ["SecoAmazonas"]],
    9: ["Pablo_2br_14ar_format", ["AR"]],
    10: ["Pablo_2br_nar_format", ["AR"]],
}
dataset = dict_dataset[9] # change
dataset

['Pablo_2br_14ar_format', ['AR']]

In [28]:
params = {
    "exp": "exp9", # Change
    "methods": ["t-gae"], # ["vgae-base", "argva-base", "vgae-line", "dgi-tran", "t-gae"],
    "data_variations": ["none"],
    "apply_transformation": False, # True or False
    "controls": dataset[1],
    "dimension": 64,
    "threshold_corr": 0.5,
    "threshold_log2": 0,
    "alpha": 0.05,
    "iterations": 1,
    "raw_data_file": dataset[0],
    "groups_id_no": ["Blank", "QC", "Std"],
    "sensitivity": False, # False: f1 (selectivity), True: f1 (selectivity), f2 (sensitivity)
    "obs": "",
    "seeds": [41, 42, 43, 44, 45, 46],
    
    "from": "python",
    "cuda": 0,
    "epochs": 500,
    "lr": 0.0001,
    "weight_decay": 1e-4,
    "patience": 10,
    "contamination": 0.1, # float in (0., 0.5)
    "n_jobs": 1, # -1 all
}

In [29]:
""" dir_path = "experiments/output"
res = sorted(os.listdir(dir_path))
n = len(res)
exp = "exp{}".format(n) """

exp = str(params["exp"])
exp

'exp9'

### Load dataset

In [30]:
# load dataset groups
if params["from"] == "python":
    df_raw = pd.read_csv("experiments/raw_data/{}.csv".format(params["raw_data_file"]), delimiter="|")
elif params["from"] == "drf":
    df_raw = pd.read_csv("{}".format(params["raw_data_file"]), delimiter="|") # from DRF
df_raw

,Alignment ID,Average Rt,Average Mz,Metabolite name,AR_1.1,AR_1.2,AR_1.3,AR_1.4,AR_1.5,AR_1.6,...,PD_2.5,PD_2.6,PD_2.7,PD_2.8,PD_2.9,PD_2.10,PD_2.11,PD_2.12,PD_2.13,PD_2.14
0,0,1,69.99951,Unknown,0.57533,-2.21576,-1.87702,-1.51884,-1.39884,-1.29892,...,0.98788,-0.84910,-0.00751,0.45284,0.43153,-0.04444,0.46331,0.02780,-1.09413,-1.07540
1,1,1,70.04025,Unknown,2.51970,0.52452,0.86264,1.22016,1.33995,1.43969,...,3.04923,2.96805,2.60501,3.06450,3.04323,2.56814,3.07496,2.64025,4.43489,2.05710
2,2,1,70.04151,Unknown,0.58098,-0.48955,-0.21483,0.07565,0.17297,0.25400,...,1.67929,1.93973,1.11480,1.48813,1.47085,1.08484,1.49663,1.14343,0.52253,1.85662
3,3,1,70.04908,Unknown,2.57051,-0.10795,0.21509,0.55668,0.67112,0.76641,...,2.38948,1.70959,1.86469,2.30371,2.28339,1.82946,2.31369,1.89836,0.67806,0.69874
4,4,1,70.06267,Unknown,1.15787,-0.15137,0.12754,0.29794,0.42245,0.52127,...,2.60041,1.39557,1.36967,1.75502,1.73727,1.33853,1.76374,1.39940,1.12760,2.77683
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5439,5439,1,732.79951,Unknown,2.21477,-0.58849,-0.38930,-0.17867,-0.10810,4.23613,...,0.97794,0.89484,0.53114,0.81967,0.80618,0.50653,0.82628,0.55450,0.10928,0.17849
5440,5440,1,748.76437,Unknown,0.72580,-1.43132,-1.05285,-0.65267,-0.51859,-0.40695,...,1.16682,1.08808,0.89559,1.39809,1.37400,0.85432,1.40992,0.93504,0.20434,0.27869
5441,5441,1,794.79590,Unknown,1.65124,-1.20428,-0.80051,-0.37358,-0.23053,-0.11143,...,1.03778,0.97033,1.21288,1.75795,1.73195,1.16780,1.77073,1.25592,0.32484,0.38565
5442,5442,1,800.81295,Unknown,1.51148,-1.03648,-0.60768,-0.15429,-0.00238,0.12410,...,1.34152,1.28640,1.55386,2.12317,2.09588,1.50710,2.13658,1.59856,0.75971,0.81518


### Format dataset

In [31]:
# apply transformation
columns_data = list(df_raw.columns)[4:]
if params["apply_transformation"]:
    print("transformation")
    for column in columns_data:
        df_raw[column] = df_raw[column].apply(lambda x: 10**x)
df_raw

,Alignment ID,Average Rt,Average Mz,Metabolite name,AR_1.1,AR_1.2,AR_1.3,AR_1.4,AR_1.5,AR_1.6,...,PD_2.5,PD_2.6,PD_2.7,PD_2.8,PD_2.9,PD_2.10,PD_2.11,PD_2.12,PD_2.13,PD_2.14
0,0,1,69.99951,Unknown,0.57533,-2.21576,-1.87702,-1.51884,-1.39884,-1.29892,...,0.98788,-0.84910,-0.00751,0.45284,0.43153,-0.04444,0.46331,0.02780,-1.09413,-1.07540
1,1,1,70.04025,Unknown,2.51970,0.52452,0.86264,1.22016,1.33995,1.43969,...,3.04923,2.96805,2.60501,3.06450,3.04323,2.56814,3.07496,2.64025,4.43489,2.05710
2,2,1,70.04151,Unknown,0.58098,-0.48955,-0.21483,0.07565,0.17297,0.25400,...,1.67929,1.93973,1.11480,1.48813,1.47085,1.08484,1.49663,1.14343,0.52253,1.85662
3,3,1,70.04908,Unknown,2.57051,-0.10795,0.21509,0.55668,0.67112,0.76641,...,2.38948,1.70959,1.86469,2.30371,2.28339,1.82946,2.31369,1.89836,0.67806,0.69874
4,4,1,70.06267,Unknown,1.15787,-0.15137,0.12754,0.29794,0.42245,0.52127,...,2.60041,1.39557,1.36967,1.75502,1.73727,1.33853,1.76374,1.39940,1.12760,2.77683
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5439,5439,1,732.79951,Unknown,2.21477,-0.58849,-0.38930,-0.17867,-0.10810,4.23613,...,0.97794,0.89484,0.53114,0.81967,0.80618,0.50653,0.82628,0.55450,0.10928,0.17849
5440,5440,1,748.76437,Unknown,0.72580,-1.43132,-1.05285,-0.65267,-0.51859,-0.40695,...,1.16682,1.08808,0.89559,1.39809,1.37400,0.85432,1.40992,0.93504,0.20434,0.27869
5441,5441,1,794.79590,Unknown,1.65124,-1.20428,-0.80051,-0.37358,-0.23053,-0.11143,...,1.03778,0.97033,1.21288,1.75795,1.73195,1.16780,1.77073,1.25592,0.32484,0.38565
5442,5442,1,800.81295,Unknown,1.51148,-1.03648,-0.60768,-0.15429,-0.00238,0.12410,...,1.34152,1.28640,1.55386,2.12317,2.09588,1.50710,2.13658,1.59856,0.75971,0.81518


In [32]:
# concat
df_join_raw = pd.concat([
    df_raw.iloc[:, :]], axis=1)
df_join_raw.set_index("Alignment ID", inplace=True)
df_join_raw

,Average Rt,Average Mz,Metabolite name,AR_1.1,AR_1.2,AR_1.3,AR_1.4,AR_1.5,AR_1.6,AR_1.7,...,PD_2.5,PD_2.6,PD_2.7,PD_2.8,PD_2.9,PD_2.10,PD_2.11,PD_2.12,PD_2.13,PD_2.14
Alignment ID,,,,,,,,,,,,,,,,,,,,,
0,1,69.99951,Unknown,0.57533,-2.21576,-1.87702,-1.51884,-1.39884,-1.29892,-1.21306,...,0.98788,-0.84910,-0.00751,0.45284,0.43153,-0.04444,0.46331,0.02780,-1.09413,-1.07540
1,1,70.04025,Unknown,2.51970,0.52452,0.86264,1.22016,1.33995,1.43969,1.52539,...,3.04923,2.96805,2.60501,3.06450,3.04323,2.56814,3.07496,2.64025,4.43489,2.05710
2,1,70.04151,Unknown,0.58098,-0.48955,-0.21483,0.07565,0.17297,0.25400,0.32363,...,1.67929,1.93973,1.11480,1.48813,1.47085,1.08484,1.49663,1.14343,0.52253,1.85662
3,1,70.04908,Unknown,2.57051,-0.10795,0.21509,0.55668,0.67112,0.76641,0.84829,...,2.38948,1.70959,1.86469,2.30371,2.28339,1.82946,2.31369,1.89836,0.67806,0.69874
4,1,70.06267,Unknown,1.15787,-0.15137,0.12754,0.29794,0.42245,0.52127,0.60354,...,2.60041,1.39557,1.36967,1.75502,1.73727,1.33853,1.76374,1.39940,1.12760,2.77683
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5439,1,732.79951,Unknown,2.21477,-0.58849,-0.38930,-0.17867,-0.10810,4.23613,-0.04934,...,0.97794,0.89484,0.53114,0.81967,0.80618,0.50653,0.82628,0.55450,0.10928,0.17849
5440,1,748.76437,Unknown,0.72580,-1.43132,-1.05285,-0.65267,-0.51859,-0.40695,-0.31103,...,1.16682,1.08808,0.89559,1.39809,1.37400,0.85432,1.40992,0.93504,0.20434,0.27869
5441,1,794.79590,Unknown,1.65124,-1.20428,-0.80051,-0.37358,-0.23053,-0.11143,-0.00910,...,1.03778,0.97033,1.21288,1.75795,1.73195,1.16780,1.77073,1.25592,0.32484,0.38565


In [33]:
# split
df_join_raw = df_join_raw.rename_axis(None)
# df_join_raw = df_join_raw.iloc[:, 2:]
df_join_raw

,Average Rt,Average Mz,Metabolite name,AR_1.1,AR_1.2,AR_1.3,AR_1.4,AR_1.5,AR_1.6,AR_1.7,...,PD_2.5,PD_2.6,PD_2.7,PD_2.8,PD_2.9,PD_2.10,PD_2.11,PD_2.12,PD_2.13,PD_2.14
0,1,69.99951,Unknown,0.57533,-2.21576,-1.87702,-1.51884,-1.39884,-1.29892,-1.21306,...,0.98788,-0.84910,-0.00751,0.45284,0.43153,-0.04444,0.46331,0.02780,-1.09413,-1.07540
1,1,70.04025,Unknown,2.51970,0.52452,0.86264,1.22016,1.33995,1.43969,1.52539,...,3.04923,2.96805,2.60501,3.06450,3.04323,2.56814,3.07496,2.64025,4.43489,2.05710
2,1,70.04151,Unknown,0.58098,-0.48955,-0.21483,0.07565,0.17297,0.25400,0.32363,...,1.67929,1.93973,1.11480,1.48813,1.47085,1.08484,1.49663,1.14343,0.52253,1.85662
3,1,70.04908,Unknown,2.57051,-0.10795,0.21509,0.55668,0.67112,0.76641,0.84829,...,2.38948,1.70959,1.86469,2.30371,2.28339,1.82946,2.31369,1.89836,0.67806,0.69874
4,1,70.06267,Unknown,1.15787,-0.15137,0.12754,0.29794,0.42245,0.52127,0.60354,...,2.60041,1.39557,1.36967,1.75502,1.73727,1.33853,1.76374,1.39940,1.12760,2.77683
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5439,1,732.79951,Unknown,2.21477,-0.58849,-0.38930,-0.17867,-0.10810,4.23613,-0.04934,...,0.97794,0.89484,0.53114,0.81967,0.80618,0.50653,0.82628,0.55450,0.10928,0.17849
5440,1,748.76437,Unknown,0.72580,-1.43132,-1.05285,-0.65267,-0.51859,-0.40695,-0.31103,...,1.16682,1.08808,0.89559,1.39809,1.37400,0.85432,1.40992,0.93504,0.20434,0.27869
5441,1,794.79590,Unknown,1.65124,-1.20428,-0.80051,-0.37358,-0.23053,-0.11143,-0.00910,...,1.03778,0.97033,1.21288,1.75795,1.73195,1.16780,1.77073,1.25592,0.32484,0.38565
5442,1,800.81295,Unknown,1.51148,-1.03648,-0.60768,-0.15429,-0.00238,0.12410,0.23278,...,1.34152,1.28640,1.55386,2.12317,2.09588,1.50710,2.13658,1.59856,0.75971,0.81518


In [34]:
# get groups name
groups_id_no = params["groups_id_no"]
groups_id = []
for item in df_join_raw.iloc[:, 3:].columns.values:
    group_id = item.split("_")[0]
    if group_id not in groups_id and group_id not in groups_id_no:
        groups_id.append(group_id)
groups_id

['AR', 'CRS', 'OSA', 'LPRD', 'SGB', 'LSNB', 'RCC', 'BC', 'BPH', 'PCa', 'PD']

In [35]:
# delete no sample columns
""" columns_delete = [columna for columna in df_join_raw.columns if columna.split("_")[0] in columns_no_sample]
df_join_raw.drop(columns_delete, axis=1, inplace=True)
df_join_raw """

' columns_delete = [columna for columna in df_join_raw.columns if columna.split("_")[0] in columns_no_sample]\ndf_join_raw.drop(columns_delete, axis=1, inplace=True)\ndf_join_raw '

In [36]:
# get subgroups names

""" def get_subgroups_id(df_join_raw, groups, by_group=False):
    dict_groups_id = {}
    for group in groups:
        # get group
        if by_group:
            dict_groups_id[group] = ["1"]
        else:
            columns = list(df_join_raw.filter(like=group).columns)
            subgroups = [item.split("{}_".format(group))[1].split(".")[0] for item in columns]
            subgroups = np.unique(subgroups)
            dict_groups_id[group] = subgroups.tolist()
    return dict_groups_id """

subgroups_id = get_subgroups_id(df_join_raw, groups_id)
subgroups_id

{'AR': ['1', '2'],
 'CRS': ['1', '2'],
 'OSA': ['1', '2'],
 'LPRD': ['1', '2'],
 'SGB': ['1', '2'],
 'LSNB': ['1', '2'],
 'RCC': ['1', '2'],
 'BC': ['1', '2'],
 'BPH': ['1', '2'],
 'PCa': ['1', '2'],
 'PD': ['1', '2']}

In [37]:
# count analtical repetitions
# df_join_raw.filter(like="AA_1.")

In [38]:
# check distribution

In [39]:
""" x = df_join_raw.iloc[2, 3:]
print(x.min(), x.max(), x.mean())
x.hist(bins=200) """

' x = df_join_raw.iloc[2, 3:]\nprint(x.min(), x.max(), x.mean())\nx.hist(bins=200) '

In [40]:
# f_join_raw.iloc[:, 5].hist(bins=100)

In [41]:
params["controls"], groups_id

(['AR'],
 ['AR', 'CRS', 'OSA', 'LPRD', 'SGB', 'LSNB', 'RCC', 'BC', 'BPH', 'PCa', 'PD'])

In [42]:
# get groups combination
groups = []
controls = params["controls"]

groups = []
for control in controls:
    for group_id in groups_id:
        if control != group_id:
            groups.append([control, group_id])
print(groups)

[['AR', 'CRS'], ['AR', 'OSA'], ['AR', 'LPRD'], ['AR', 'SGB'], ['AR', 'LSNB'], ['AR', 'RCC'], ['AR', 'BC'], ['AR', 'BPH'], ['AR', 'PCa'], ['AR', 'PD']]


### Create folders

In [43]:
# create experiments folder
try: 
    os.mkdir("experiments/output/{}".format(exp))
    os.mkdir("experiments/output/{}/correlations".format(exp))
    os.mkdir("experiments/output/{}/preprocessing".format(exp))
    os.mkdir("experiments/output/{}/preprocessing/edges".format(exp))
    os.mkdir("experiments/output/{}/preprocessing/graphs_data".format(exp))
    os.mkdir("experiments/output/{}/loss".format(exp))
    os.mkdir("experiments/output/{}/node_embeddings".format(exp))
    os.mkdir("experiments/output/{}/common_nodes".format(exp))
    os.mkdir("experiments/output/{}/filter_raw".format(exp))
    os.mkdir("experiments/output/{}/plots".format(exp))
except OSError as error: 
    print(error)

### Save dataset and parameters

In [44]:
# save dataset
df_join_raw.to_csv("experiments/input/{}_raw.csv".format(exp), index=True)

# save parameters
parameters = {
    "exp": exp,
    "methods": params["methods"],
    "data_variations": params["data_variations"],
    "apply_transformation": params["apply_transformation"],
    "controls": params["controls"],
    "dimension": params["dimension"],
    "threshold_corr": params["threshold_corr"],
    "threshold_log2": params["threshold_log2"],
    "alpha": params["alpha"],
    "iterations": params["iterations"],
    "raw_data_file": params["raw_data_file"],
    "groups_id": groups_id,
    "subgroups_id": subgroups_id,
    "groups": groups,
    "groups_id_no": params["groups_id_no"],
    "sensitivity": params["sensitivity"],
    
    "from": params["from"],
    "cuda": params["cuda"],
    "epochs": params["epochs"],
    "lr": params["lr"],
    "weight_decay": params["weight_decay"],
    "patience": params["patience"],
    "contamination": params["contamination"],
    "n_jobs": params["n_jobs"],

    "seeds": params["seeds"],
    "obs": params["obs"]
}

with open("experiments/output/{}/parameters.json".format(exp), "w") as outfile:
    json.dump(parameters, outfile, indent=4)

In [45]:
experiments = {
    "exp": exp
}

with open("exp.json".format(experiments), "w") as outfile:
    json.dump(experiments, outfile, indent=4)

In [46]:
# return exp